# Vaelo — Financial Health Snapshot Generation Pipeline

Pipeline 3 of 3 (Financial Health Snapshot). Same design principle as Pipelines 1 and 2 —
**deterministic and formula-driven, no LLM, no trained model.** Per SRS FR-4, this is the
lightweight, low-cost, recurring-check-in offering — not a replacement for the Valuation
or Deal Feasibility reports, a companion to them.

**A deliberate design decision, made explicitly rather than defaulted into:** this pipeline
outputs **four independent sub-scores**, not one collapsed composite score. A single
0-100 "health score" would require picking arbitrary weights between four genuinely
different risk dimensions (e.g. "why does cash runway count for 30% and not 25%?") —
exactly the kind of unexplainable judgment call the "no black box, always traceable"
positioning is built to avoid. Four separately-explainable numbers sidestep that problem
entirely, at the cost of being slightly less quotable than a single score.

**Pipeline stages:**
1. Required documents & data
2. Structured intake schema
3. Sub-score 1 — Liquidity (Current Ratio)
4. Sub-score 2 — Expense Growth Rate
5. Sub-score 3 — Cash Runway
6. Sub-score 4 — Revenue Volatility
7. Orchestration — run all four
8. Report generator (templated)
9. End-to-end example run
10. Stress test — confirming a genuinely unhealthy business scores accordingly

## 1. Required Documents & Data

| Item | Why it's needed |
|---|---|
| Current Assets & Current Liabilities (latest balance sheet) | Current Ratio (liquidity) |
| 2+ years of Revenue and Operating Expenses | Expense growth rate vs. revenue growth rate |
| Cash & Equivalents, and average monthly operating cash burn (or net cash flow) | Cash runway |
| 3+ years of historical annual Revenue | Revenue volatility |

This is intentionally less data than Pipelines 1 or 2 need — that's the point of this
being the lightweight, faster, cheaper snapshot rather than a full valuation or deal
analysis.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date
import statistics

## 2. Structured Intake Schema

`HealthSnapshotRequest` is the single object a CA submits — same intake pattern as
Pipelines 1 and 2, scaled down to what this lighter report actually needs.

In [ ]:
@dataclass
class SnapshotMeta:
    """Who this snapshot is for and why."""
    client_name: str
    ca_firm_name: str
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class LiquidityInputs:
    """Latest balance sheet position, Rs Cr."""
    current_assets: float
    current_liabilities: float


@dataclass
class ExpenseGrowthInputs:
    """Most recent two full years, Rs Cr. Oldest to newest."""
    revenue_prior_year: float
    revenue_current_year: float
    operating_expenses_prior_year: float
    operating_expenses_current_year: float


@dataclass
class CashRunwayInputs:
    """Rs Cr. If the business is net cash-generative (not burning cash),
    set monthly_net_cash_flow >= 0 and the engine reports runway as N/A —
    a business that generates cash doesn't have a 'burn runway' to measure."""
    cash_and_equivalents: float
    monthly_net_cash_flow: float          # negative if burning cash, positive if generating


@dataclass
class RevenueVolatilityInputs:
    """3+ years of historical annual revenue, Rs Cr, oldest to newest.
    More years gives a more reliable volatility read — 3 is the practical minimum."""
    historical_revenue: List[float]


@dataclass
class HealthSnapshotRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: SnapshotMeta
    liquidity: LiquidityInputs
    expense_growth: ExpenseGrowthInputs
    cash_runway: CashRunwayInputs
    revenue_volatility: RevenueVolatilityInputs

## 3. Sub-Score 1 — Liquidity (Current Ratio)

Current Ratio = Current Assets / Current Liabilities. Score bands below are documented
thresholds, not learned weights — a CA can see exactly why a business landed at a given
score and challenge the thresholds directly if their judgment differs.

Note the top band deliberately does not keep climbing forever: a very high current ratio
(>3.0) isn't purely "better" — it often means idle, poorly-deployed capital, so the score
is capped and the report says so explicitly rather than implying "higher is always best."

In [ ]:
def score_liquidity(inputs: LiquidityInputs) -> dict:
    """Current Ratio -> 0-100 score with a plain-language label."""
    if inputs.current_liabilities <= 0:
        current_ratio = float("inf")
    else:
        current_ratio = inputs.current_assets / inputs.current_liabilities

    if current_ratio < 1.0:
        score = max(0, current_ratio * 40)                      # 0.0 -> 0, 1.0 -> 40
        label = "Weak"
        note = "Current liabilities exceed current assets — a real short-term liquidity risk."
    elif current_ratio < 1.5:
        score = 40 + (current_ratio - 1.0) / 0.5 * 25            # 1.0 -> 40, 1.5 -> 65
        label = "Adequate"
        note = "Liquidity is workable but has limited cushion."
    elif current_ratio <= 3.0:
        score = 65 + (current_ratio - 1.5) / 1.5 * 25            # 1.5 -> 65, 3.0 -> 90
        label = "Strong"
        note = "Healthy short-term liquidity cushion."
    else:
        score = 85                                                # capped, not climbing further
        label = "Excess (possible idle capital)"
        note = "Very high current ratio — worth checking whether working capital is being deployed efficiently rather than sitting idle."

    return {
        "current_ratio": round(current_ratio, 2) if current_ratio != float("inf") else None,
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

## 4. Sub-Score 2 — Expense Growth Rate

Compares expense growth against revenue growth, not expense growth in isolation. Expenses
growing *slower* than revenue is healthy (improving operating leverage); expenses
outgrowing revenue is a margin-compression warning sign, regardless of whether the
absolute expense growth number looks large or small on its own.

In [ ]:
def score_expense_growth(inputs: ExpenseGrowthInputs) -> dict:
    """Expense growth rate vs. revenue growth rate -> 0-100 score."""
    revenue_growth = (
        (inputs.revenue_current_year - inputs.revenue_prior_year) / inputs.revenue_prior_year
        if inputs.revenue_prior_year else 0
    )
    expense_growth = (
        (inputs.operating_expenses_current_year - inputs.operating_expenses_prior_year)
        / inputs.operating_expenses_prior_year
        if inputs.operating_expenses_prior_year else 0
    )
    spread = expense_growth - revenue_growth   # negative = expenses growing slower than revenue (good)

    if spread <= -0.05:
        score = 100
        label = "Excellent"
        note = "Expenses are growing meaningfully slower than revenue — operating leverage is improving."
    elif spread <= 0:
        score = 75 + (abs(spread) / 0.05) * 25 if spread != 0 else 75   # 0 -> 75, -0.05 -> 100
        label = "Good"
        note = "Expenses are keeping pace with or growing slightly slower than revenue."
    elif spread <= 0.05:
        score = 40 + (1 - spread / 0.05) * 35                           # 0 -> 75, 0.05 -> 40
        label = "Caution"
        note = "Expenses are growing modestly faster than revenue — worth monitoring."
    else:
        score = max(0, 40 - (spread - 0.05) * 200)                      # steep drop past 5pt spread
        label = "Concerning"
        note = "Expenses are significantly outgrowing revenue — margin compression risk."

    return {
        "revenue_growth_pct": round(revenue_growth * 100, 1),
        "expense_growth_pct": round(expense_growth * 100, 1),
        "spread_pct": round(spread * 100, 1),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

## 5. Sub-Score 3 — Cash Runway

Runway in months = Cash & Equivalents / monthly cash burn — but only meaningful if the
business is actually burning cash. A business with positive net cash flow doesn't have
a "runway" in the depletion sense, so that case is reported as N/A with the maximum
score rather than a fabricated number.

In [ ]:
def score_cash_runway(inputs: CashRunwayInputs) -> dict:
    """Months of runway at current burn rate -> 0-100 score."""
    if inputs.monthly_net_cash_flow >= 0:
        return {
            "runway_months": None,
            "score": 100.0,
            "label": "Cash-generative",
            "note": "Business is generating positive net cash flow — no burn runway to measure.",
        }

    monthly_burn = abs(inputs.monthly_net_cash_flow)
    runway_months = inputs.cash_and_equivalents / monthly_burn if monthly_burn else float("inf")

    if runway_months < 3:
        score = runway_months / 3 * 20                    # 0 -> 0, 3 -> 20
        label = "Critical"
        note = "Less than 3 months of runway at current burn rate — immediate attention required."
    elif runway_months < 6:
        score = 20 + (runway_months - 3) / 3 * 30           # 3 -> 20, 6 -> 50
        label = "Weak"
        note = "Runway is short — limited room to absorb a slow quarter or delayed receivables."
    elif runway_months < 12:
        score = 50 + (runway_months - 6) / 6 * 25           # 6 -> 50, 12 -> 75
        label = "Adequate"
        note = "Reasonable buffer, though worth monitoring if burn increases."
    else:
        score = min(75 + (runway_months - 12) / 6 * 25, 100)  # 12 -> 75, 18+ -> 100
        label = "Strong"
        note = "Healthy cash buffer relative to current burn rate."

    return {
        "runway_months": round(runway_months, 1),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

## 6. Sub-Score 4 — Revenue Volatility

Uses the coefficient of variation (standard deviation / mean) of historical annual
revenue — a standard, simple measure of how predictable the top line has been. Lower
volatility scores higher, since predictability matters to a lender or acquirer
independent of the growth rate itself (a volatile 20%-average-growth business is
riskier than a steady 8%-average-growth one, even though the raw growth number looks
better).

In [ ]:
def score_revenue_volatility(inputs: RevenueVolatilityInputs) -> dict:
    """Coefficient of variation of historical revenue -> 0-100 score."""
    revenues = inputs.historical_revenue
    if len(revenues) < 3:
        raise ValueError(
            f"Revenue volatility requires at least 3 years of historical revenue, "
            f"got {len(revenues)} — this is a hard minimum for a meaningful read, not a suggestion."
        )

    mean_rev = statistics.mean(revenues)
    stdev_rev = statistics.stdev(revenues)
    cv = stdev_rev / mean_rev if mean_rev else float("inf")

    if cv < 0.05:
        score = 100 - (cv / 0.05) * 10                      # 0 -> 100, 0.05 -> 90
        label = "Very stable"
        note = "Revenue has been highly predictable year over year."
    elif cv < 0.15:
        score = 90 - (cv - 0.05) / 0.10 * 20                 # 0.05 -> 90, 0.15 -> 70
        label = "Stable"
        note = "Revenue shows normal, manageable year-to-year variation."
    elif cv < 0.30:
        score = 70 - (cv - 0.15) / 0.15 * 30                 # 0.15 -> 70, 0.30 -> 40
        label = "Variable"
        note = "Revenue swings meaningfully year to year — worth understanding the cause (seasonality, customer concentration, etc.)."
    else:
        score = max(0, 40 - (cv - 0.30) * 100)
        label = "Highly volatile"
        note = "Revenue is highly unpredictable — this materially increases risk independent of the average growth rate."

    return {
        "coefficient_of_variation": round(cv, 3),
        "mean_revenue": round(mean_rev, 2),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

## 7. Orchestration — Run All Four

One function calling all four sub-scores and returning a single dict — the report
generator consumes this directly. Deliberately **no composite score is computed here** —
see the design note in Section 1.

In [ ]:
def calculate_health_snapshot(req: HealthSnapshotRequest) -> dict:
    """Runs all four independent sub-scores. No composite/weighted score by design."""
    liquidity = score_liquidity(req.liquidity)
    expense_growth = score_expense_growth(req.expense_growth)
    cash_runway = score_cash_runway(req.cash_runway)
    revenue_volatility = score_revenue_volatility(req.revenue_volatility)

    # Flag any sub-score below a "worth watching" threshold — this is a highlight,
    # not a collapsed composite number.
    flags = []
    for name, result in [
        ("Liquidity", liquidity),
        ("Expense Growth", expense_growth),
        ("Cash Runway", cash_runway),
        ("Revenue Volatility", revenue_volatility),
    ]:
        if result["score"] < 40:
            flags.append(f"{name}: {result['label']} ({result['score']}/100) — {result['note']}")

    return {
        "liquidity": liquidity,
        "expense_growth": expense_growth,
        "cash_runway": cash_runway,
        "revenue_volatility": revenue_volatility,
        "flags": flags,
    }

## 8. Report Generator (Templated, Not Generative)

Same principle as Pipelines 1 and 2 — plain string formatting pulling from the
calculated dict, no LLM, every line traceable. Presents all four scores side by side,
with no single number pretending to summarize the whole business.

In [ ]:
def build_health_snapshot_report(req: HealthSnapshotRequest, result: dict) -> str:
    """Templated report generation — plain string formatting, no LLM."""
    lines = []
    lines.append(f"FINANCIAL HEALTH SNAPSHOT — {req.meta.client_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)
    lines.append("This snapshot presents four independent measures of financial health.")
    lines.append("They are shown separately, not collapsed into a single score, so each")
    lines.append("dimension can be reviewed and explained on its own.")

    liq = result["liquidity"]
    lines.append("\n--- 1. LIQUIDITY (Current Ratio) ---")
    lines.append(f"Current Ratio: {liq['current_ratio']}")
    lines.append(f"Score: {liq['score']}/100  ({liq['label']})")
    lines.append(f"Note: {liq['note']}")

    exp = result["expense_growth"]
    lines.append("\n--- 2. EXPENSE GROWTH RATE ---")
    lines.append(f"Revenue growth: {exp['revenue_growth_pct']:+.1f}%   Expense growth: {exp['expense_growth_pct']:+.1f}%")
    lines.append(f"Spread (expense - revenue growth): {exp['spread_pct']:+.1f} pts")
    lines.append(f"Score: {exp['score']}/100  ({exp['label']})")
    lines.append(f"Note: {exp['note']}")

    cash = result["cash_runway"]
    lines.append("\n--- 3. CASH RUNWAY ---")
    if cash["runway_months"] is None:
        lines.append("Runway: N/A (cash-generative)")
    else:
        lines.append(f"Runway: {cash['runway_months']} months")
    lines.append(f"Score: {cash['score']}/100  ({cash['label']})")
    lines.append(f"Note: {cash['note']}")

    vol = result["revenue_volatility"]
    lines.append("\n--- 4. REVENUE VOLATILITY ---")
    lines.append(f"Coefficient of variation: {vol['coefficient_of_variation']}")
    lines.append(f"Score: {vol['score']}/100  ({vol['label']})")
    lines.append(f"Note: {vol['note']}")

    lines.append("\n--- ITEMS TO WATCH ---")
    if result["flags"]:
        for f in result["flags"]:
            lines.append(f"  - {f}")
    else:
        lines.append("No sub-score fell below the review threshold (40/100).")

    lines.append("\n" + "=" * 60)
    lines.append("This snapshot is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied. No composite score is")
    lines.append("computed by design — see the note at the top of this pipeline.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

## 9. End-to-End Example Run

Illustrative figures only — a broadly healthy small business.

In [ ]:
# --- Example HealthSnapshotRequest (replace with real client data) ---

example_snapshot = HealthSnapshotRequest(
    meta=SnapshotMeta(
        client_name="Example SME Pvt Ltd",
        ca_firm_name="Example & Associates",
        sector="Manufacturing",
    ),
    liquidity=LiquidityInputs(
        current_assets=6.2,
        current_liabilities=3.4,
    ),
    expense_growth=ExpenseGrowthInputs(
        revenue_prior_year=9.2,
        revenue_current_year=10.5,
        operating_expenses_prior_year=7.6,
        operating_expenses_current_year=8.3,
    ),
    cash_runway=CashRunwayInputs(
        cash_and_equivalents=2.1,
        monthly_net_cash_flow=-0.12,     # burning modestly
    ),
    revenue_volatility=RevenueVolatilityInputs(
        historical_revenue=[8.0, 9.2, 10.5, 10.1, 11.4],
    ),
)

result = calculate_health_snapshot(example_snapshot)
report = build_health_snapshot_report(example_snapshot, result)
print(report)

## 10. Stress Test — Confirming an Unhealthy Business Scores Accordingly

Same discipline used in Pipeline 2: a clean example only proves the happy path. This
run deliberately uses a genuinely distressed business — weak liquidity, expenses
outgrowing revenue, a few months of cash runway, and volatile revenue — and asserts
that every sub-score lands in its correspondingly weak band, and that the flag list
actually catches all of them.

In [ ]:
# --- Stress test: a genuinely unhealthy business ---

stress_snapshot = HealthSnapshotRequest(
    meta=SnapshotMeta(
        client_name="Distressed Example Pvt Ltd",
        ca_firm_name="Example & Associates",
        sector="Retail",
    ),
    liquidity=LiquidityInputs(
        current_assets=2.0,
        current_liabilities=3.0,           # ratio < 1.0 -> Weak
    ),
    expense_growth=ExpenseGrowthInputs(
        revenue_prior_year=10.0,
        revenue_current_year=10.2,          # ~2% revenue growth
        operating_expenses_prior_year=8.0,
        operating_expenses_current_year=9.6,  # ~20% expense growth -> big positive spread
    ),
    cash_runway=CashRunwayInputs(
        cash_and_equivalents=0.6,
        monthly_net_cash_flow=-0.30,        # burning fast -> 2 months runway
    ),
    revenue_volatility=RevenueVolatilityInputs(
        historical_revenue=[10.0, 4.0, 14.0, 3.5, 11.0],   # genuinely erratic, CV well above 0.30
    ),
)

stress_result = calculate_health_snapshot(stress_snapshot)
stress_report = build_health_snapshot_report(stress_snapshot, stress_result)
print(stress_report)

In [ ]:
# --- Confirm every sub-score landed in its expected weak band, and flags caught them ---

assert stress_result["liquidity"]["label"] == "Weak", "Expected Weak liquidity label"
assert stress_result["liquidity"]["score"] < 40, "Expected liquidity score below review threshold"

assert stress_result["expense_growth"]["spread_pct"] > 5, "Expected a large positive expense/revenue growth spread"
assert stress_result["expense_growth"]["label"] == "Concerning", "Expected Concerning expense growth label"

assert stress_result["cash_runway"]["runway_months"] < 3, "Expected critical (<3mo) runway"
assert stress_result["cash_runway"]["label"] == "Critical"

assert stress_result["revenue_volatility"]["coefficient_of_variation"] > 0.15, "Expected meaningfully volatile revenue"

assert len(stress_result["flags"]) == 4, (
    f"Expected all four sub-scores to be flagged for this deliberately distressed "
    f"scenario, got {len(stress_result['flags'])}"
)

print("All stress-test assertions passed — every sub-score and the flag list behaved as expected under distressed inputs.")

## Next Steps

- **Replace placeholder data** with real client figures before using this for an actual
  engagement, same as Pipelines 1 and 2.
- **The 40/100 flag threshold and all scoring band breakpoints are defaults** — tune them
  per sector or the CA's own risk appetite if their judgment differs; the point of
  documenting every threshold explicitly is that they're meant to be challenged and
  adjusted, not treated as fixed truth.
- **This is intentionally the lightest pipeline** — no PDF/WhatsApp delivery build-out
  yet, per the same founder-in-the-loop v1 scope as Pipelines 1 and 2 (SRS FR-6).
- With all three pipelines now built and stress-tested, the next real step is running
  each against actual or realistic full client scenarios — not more building — before
  taking these to the CA conversations that determine whether any of this is worth
  automating further.